In [514]:
import re 
import json
import os
import sys
from functools import reduce
from collections import Counter
import numpy as np
import ast
from copy import deepcopy
from collections import Counter

In [515]:
BENCHMARK = 'nq'
EXP_NAME = 'natural_qwen257b_medium_mixture(#3e65aec8)(v2.3.7)'
MAX_SEARCH_STEPS = 8

LOAD_MEDIUM_REASEONER_LOG_PATH = f"../../../../pai2_mainexps/{BENCHMARK}/{EXP_NAME}/log/qa/kg_reasoner/medium/main/.log"
LOAD_QUERYPREP_LOG_PATH = f"../../../../pai2_mainexps/{BENCHMARK}/{EXP_NAME}/log/qa/query_preprocessing/main/.log"
SAVE_LLMCALL_COUNTERS_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/llmcalls_counters.json"
SAVE_LLMCALL_STATISTICS_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/llmcalls_statistics.json"
SAVE_SUBQUESTIONS_STATISTICS_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/subquestions_statistics.json"

MEDIUM_REASONER_LLMTASKS_TEMPLATE = {
    'answer_classify_solver': 0, # 1
    'casual_answer_gen_solver': 0, # 3
    'strict_answer_gen_solver': 0, # 4

    'clueanswers_summarizer_solver': 0, # 1
    'clueanswer_generator_solver': 0, # 2
    'cluequeries_generator_solver': 0, # 2
    'entities_extractor_solver': 0, # 1

    'searchplan_enhance_classify_solver': 0, # 1
    'plan_enhancing_solver': 0, # 1
    'plan_initialing_solver': 1 # FIXED
}

In [516]:
with open(LOAD_MEDIUM_REASEONER_LOG_PATH, 'r', encoding='utf-8') as fd:
    RAW_LOG = fd.read()

### Gathering LLM-calls information

In [517]:
# алгоритм
# 1. разбить лог по вопросам
# 2. отфильтровать вопросы, которые были закешированы
# 3. определить количество шагов плана поиска для каждого вопроса
# 4. определить количество clue-вопросов, сгенерированных для каждого шага плана

In [518]:
START_QUERY_HANDLING_LINE = "START MEDIUM KG-REASONING..."
START_SEARCHSTEP_HANDLING_LINE = "STAGE#1.1 - SEARCH PLAN INITING/ENHANCING"
START_CLUEQUERIESGENERATION_HANDLING_LINE = "STAGE#2.2 - CLUE-QUERIES GENERATION"

In [519]:
LLMCALL_COUNTERS = {

}

LOG_POINT_NAMES = {
    'start_query_handling': "SQH",
    'start_searchstep_handling': "SSSH",
    'start_cluequeries_generation': "SCQG"
}

In [520]:
MAIN_LOG_POINTS = []
LOG_LINES = RAW_LOG.split("\n")
for idx, log_line in enumerate(LOG_LINES):
    if log_line.endswith(START_QUERY_HANDLING_LINE):
        query_hash = LOG_LINES[idx+1].split(" ")[-2]
        MAIN_LOG_POINTS.append((idx, LOG_POINT_NAMES['start_query_handling'], query_hash))
    elif log_line.endswith(START_SEARCHSTEP_HANDLING_LINE):
        MAIN_LOG_POINTS.append((idx, LOG_POINT_NAMES['start_searchstep_handling']))
    elif log_line.endswith(START_CLUEQUERIESGENERATION_HANDLING_LINE):
        generated_clue_queries = int(LOG_LINES[idx+2].split(" ")[-1])
        MAIN_LOG_POINTS.append((idx, LOG_POINT_NAMES['start_cluequeries_generation'], generated_clue_queries))

In [521]:
handled_questions = list(filter(lambda item: item[1] == LOG_POINT_NAMES['start_query_handling'], MAIN_LOG_POINTS))
print(len(handled_questions))
unique_questions = set(list(map(lambda item: item[2], handled_questions)))
print(len(unique_questions))

1121
1121


In [522]:
current_query_idx = -1
current_query_hash = None
current_llmcalls_counter = None
for current_point in MAIN_LOG_POINTS:
    if current_point[1] == LOG_POINT_NAMES['start_query_handling']:
        if current_llmcalls_counter is not None:
            current_llmcalls_counter['searchplan_enhance_classify_solver'] -= 1
            current_llmcalls_counter['plan_enhancing_solver'] -= 1

            if current_llmcalls_counter['clueanswers_summarizer_solver'] == MAX_SEARCH_STEPS:
                current_llmcalls_counter['strict_answer_gen_solver'] += 1
            else:
                current_llmcalls_counter['casual_answer_gen_solver'] += 1

            LLMCALL_COUNTERS[current_query_idx] = deepcopy(current_llmcalls_counter)

        current_query_idx += 1
        current_llmcalls_counter = deepcopy(MEDIUM_REASONER_LLMTASKS_TEMPLATE)
        #current_llmcalls_counter['question_hash'] = current_point[2]

    elif current_point[1] == LOG_POINT_NAMES['start_searchstep_handling']:
        current_llmcalls_counter['entities_extractor_solver'] += 1
        current_llmcalls_counter['clueanswers_summarizer_solver'] += 1

        current_llmcalls_counter['answer_classify_solver'] += 1

        current_llmcalls_counter['searchplan_enhance_classify_solver'] += 1
        current_llmcalls_counter['plan_enhancing_solver'] += 1

    elif current_point[1] == LOG_POINT_NAMES['start_cluequeries_generation']:
        current_llmcalls_counter['cluequeries_generator_solver'] += current_point[2]
        current_llmcalls_counter['clueanswer_generator_solver'] += current_point[2]
        
    else:
        raise ValueError

if current_llmcalls_counter is not None:
    current_llmcalls_counter['searchplan_enhance_classify_solver'] -= 1
    current_llmcalls_counter['plan_enhancing_solver'] -= 1

    if current_llmcalls_counter['clueanswers_summarizer_solver'] == MAX_SEARCH_STEPS:
        current_llmcalls_counter['strict_answer_gen_solver'] += 1
    else:
        current_llmcalls_counter['casual_answer_gen_solver'] += 1

    LLMCALL_COUNTERS[current_query_idx] = deepcopy(current_llmcalls_counter)

In [523]:
len(LLMCALL_COUNTERS.keys())

1121

In [524]:
LLMCALL_COUNTERS

{0: {'answer_classify_solver': 1,
  'casual_answer_gen_solver': 1,
  'strict_answer_gen_solver': 0,
  'clueanswers_summarizer_solver': 1,
  'clueanswer_generator_solver': 4,
  'cluequeries_generator_solver': 4,
  'entities_extractor_solver': 1,
  'searchplan_enhance_classify_solver': 0,
  'plan_enhancing_solver': 0,
  'plan_initialing_solver': 1},
 1: {'answer_classify_solver': 1,
  'casual_answer_gen_solver': 1,
  'strict_answer_gen_solver': 0,
  'clueanswers_summarizer_solver': 1,
  'clueanswer_generator_solver': 4,
  'cluequeries_generator_solver': 4,
  'entities_extractor_solver': 1,
  'searchplan_enhance_classify_solver': 0,
  'plan_enhancing_solver': 0,
  'plan_initialing_solver': 1},
 2: {'answer_classify_solver': 1,
  'casual_answer_gen_solver': 1,
  'strict_answer_gen_solver': 0,
  'clueanswers_summarizer_solver': 1,
  'clueanswer_generator_solver': 8,
  'cluequeries_generator_solver': 8,
  'entities_extractor_solver': 1,
  'searchplan_enhance_classify_solver': 0,
  'plan_enha

In [525]:
with open(SAVE_LLMCALL_COUNTERS_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(LLMCALL_COUNTERS, ensure_ascii=False, indent=2))

### Calculating Statistics

In [526]:
LLMCALLS_STATISTICS  = {
    'summary': None,

    'answer_classify_solver': None,
    'casual_answer_gen_solver': None,
    'strict_answer_gen_solver': None,

    'clueanswers_summarizer_solver': None,
    'clueanswer_generator_solver': None,
    'cluequeries_generator_solver': None,
    'entities_extractor_solver': None,

    'searchplan_enhance_classify_solver': None,
    'plan_enhancing_solver': None,
    'plan_initialing_solver': None # FIXED
}

In [527]:
for llmtask_name in MEDIUM_REASONER_LLMTASKS_TEMPLATE.keys():
    llmtask_counters = []
    for query_idx, llmcall_counters in LLMCALL_COUNTERS.items():
        llmtask_counters.append(llmcall_counters[llmtask_name])

    LLMCALLS_STATISTICS[llmtask_name] = {
        'count': len(llmtask_counters),
        'max': max(llmtask_counters),
        'mean': round(np.mean(llmtask_counters),3),
        'median': np.median(llmtask_counters),
        'min': min(llmtask_counters),
        'std': round(np.std(llmtask_counters),3),
        'sum': sum(llmtask_counters)
    }

llmcalls_per_query = []
for query_idx, llmcall_counters in LLMCALL_COUNTERS.items():
    llmcalls_per_query.append(sum(list(llmcall_counters.values())))

LLMCALLS_STATISTICS['summary'] = {
    'count': len(llmcalls_per_query),
    'max': max(llmcalls_per_query),
    'mean': round(float(np.mean(llmcalls_per_query)),3),
    'median': float(np.median(llmcalls_per_query)),
    'min': min(llmcalls_per_query),
    'std': round(float(np.std(llmcalls_per_query)),3),
    'sum': sum(llmcalls_per_query)
}

In [528]:
print(LLMCALLS_STATISTICS['summary'])

{'count': 1121, 'max': 168, 'mean': 30.014, 'median': 21.0, 'min': 9, 'std': 35.148, 'sum': 33646}


In [529]:
with open(SAVE_LLMCALL_STATISTICS_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(LLMCALLS_STATISTICS, ensure_ascii=False, indent=2))

### If PAI-2 had enabled query-preprocessing stage with decomposition-operation... 

In [530]:
with open(LOAD_QUERYPREP_LOG_PATH, 'r', encoding='utf-8') as fd:
    RAW_LOG = fd.read()

In [531]:
START_QUERYDECOMPOSITION_HANDLING_LINE = "Разбиение запроса на независимые части (простые запросы)..."

In [532]:
MAIN_LOG_POINTS = []
LOG_LINES = RAW_LOG.split("\n")
for idx, log_line in enumerate(LOG_LINES):
    if log_line.endswith(START_QUERYDECOMPOSITION_HANDLING_LINE):
        subquestions = ast.literal_eval(LOG_LINES[idx+1].split("RESULT: ")[1].strip())
        MAIN_LOG_POINTS.append((idx, len(subquestions) if subquestions is not None else 0))

In [533]:
subquestions_counters = list(map(lambda item: item[1], MAIN_LOG_POINTS))

if len(subquestions_counters) > 0 :
    SUBQUESTIONS_STATISTICS = {
        'count': len(subquestions_counters),
        'max': max(subquestions_counters),
        'mean': round(float(np.mean(subquestions_counters)),3),
        'median': float(np.median(subquestions_counters)),
        'min': min(subquestions_counters),
        'std': round(float(np.std(subquestions_counters)),3),
        'sum': sum(subquestions_counters),
        'frequency': Counter(subquestions_counters)
    }
else:
    SUBQUESTIONS_STATISTICS = {
        'count': len(subquestions_counters),
        'max': -1,
        'mean': -1,
        'median': -1,
        'min': -1,
        'std': -1,
        'sum': -1,
        'frequency': None
    }

In [534]:
Counter(subquestions_counters)

Counter({1: 816, 2: 68, 3: 12, 4: 4})

In [535]:
SUBQUESTIONS_STATISTICS

{'count': 900,
 'max': 4,
 'mean': 1.116,
 'median': 1.0,
 'min': 1,
 'std': 0.394,
 'sum': 1004,
 'frequency': Counter({1: 816, 2: 68, 3: 12, 4: 4})}

In [536]:
with open(SAVE_SUBQUESTIONS_STATISTICS_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(SUBQUESTIONS_STATISTICS, ensure_ascii=False, indent=2))